In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("task4-analytics")
         .enableHiveSupport()
         .getOrCreate())

DB = "project"

In [ ]:
chk = spark.sql(f"""
    SELECT COUNT(*) AS n, COUNT(trade_date) AS n_date, COUNT(ticker) AS n_ticker
    FROM {DB}.prices_cleaned
""").collect()[0]
print(chk)
assert chk.n == chk.n_date == chk.n_ticker and chk.n > 0, \
    "prices_cleaned has NULL trade_date/ticker: column names don't match the parquet files"

In [ ]:
q1_sql = f"""
WITH base AS (
  SELECT ticker,
         to_date(trade_date)    AS d,
         CAST(high AS DOUBLE)   AS h,
         CAST(low AS DOUBLE)    AS l,
         CAST(close AS DOUBLE)  AS c,
         CAST(volume AS DOUBLE) AS v
  FROM {DB}.prices_cleaned
  WHERE status = 'accepted'
),
ratio AS (
  SELECT ticker, d, (h - l) / c AS day_range,
         v / AVG(v) OVER (PARTITION BY ticker ORDER BY d
                          ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING) AS vol_ratio,
         COUNT(*) OVER (PARTITION BY ticker ORDER BY d
                        ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING) AS n_hist
  FROM base
)
SELECT CASE WHEN vol_ratio < 1.5 THEN '1_normal (<1.5x)'
            WHEN vol_ratio < 2.5 THEN '2_high (1.5-2.5x)'
            ELSE '3_very_high (>2.5x)' END AS bucket,
       COUNT(*)                          AS n_rows,
       AVG(day_range)                    AS avg_day_range,
       percentile_approx(day_range, 0.5) AS median_day_range
FROM ratio
WHERE n_hist = 20 AND vol_ratio IS NOT NULL
GROUP BY CASE WHEN vol_ratio < 1.5 THEN '1_normal (<1.5x)'
              WHEN vol_ratio < 2.5 THEN '2_high (1.5-2.5x)'
              ELSE '3_very_high (>2.5x)' END
ORDER BY bucket
"""

q1_result = spark.sql(q1_sql)
q1_result.show(truncate=False)

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("task4/outputs", exist_ok=True)

# The result has only 3 rows, so it is safe to turn into pandas for plotting
q1_pdf = q1_result.toPandas()
q1_pdf.to_csv("task4/outputs/q1_volume_buckets.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(q1_pdf["bucket"], q1_pdf["avg_day_range"] * 100)

# Write the group size on top of each bar
for bar, n in zip(bars, q1_pdf["n_rows"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"n = {n:,}", ha="center", va="bottom")

ax.set_xlabel("Volume vs the stock's own previous 20-day average")
ax.set_ylabel("Average daily range, (high - low) / close (%)")
ax.set_title("Q1: Daily price range by volume bucket")
plt.tight_layout()
plt.savefig("task4/outputs/q1_chart.png", dpi=150)
plt.show()

In [ ]:
PRED_PATH = "task3_outputs/reference_test_predictions"   # on EMR, change to the HDFS path

spark.read.parquet(PRED_PATH).createOrReplaceTempView("predictions")
print("prediction rows (before removing duplicates):", spark.table("predictions").count())

In [ ]:
# Q2 SQL is written here so the notebook does not need any extra file.
# A readable copy is in task4/hive/q2_model_errors.sql

q2_view_sql = """
CREATE OR REPLACE TEMPORARY VIEW q2_base AS
WITH pred AS (
  SELECT DISTINCT symbol, CAST(anchor_date AS STRING) AS anchor_date, prediction, label
  FROM predictions
)
SELECT p.symbol,
       substr(p.anchor_date, 1, 7)  AS month,
       p.prediction,
       p.label,
       w.vol_12w                    AS baseline,
       NTILE(3) OVER (ORDER BY w.vol_52w) AS grp
FROM pred p
JOIN project.weekly_all w
  ON p.symbol = w.ticker
 AND p.anchor_date = CAST(w.anchor_date AS STRING)
"""

q2_month_sql = """
SELECT month,
       COUNT(*)                              AS n,
       SQRT(AVG(POW(prediction - label, 2))) AS rmse_model,
       SQRT(AVG(POW(baseline   - label, 2))) AS rmse_base,
       AVG(ABS(prediction - label))          AS mae_model,
       AVG(ABS(baseline   - label))          AS mae_base
FROM q2_base
GROUP BY month
ORDER BY month
"""

q2_group_sql = """
SELECT CASE grp WHEN 1 THEN '1_calm' WHEN 2 THEN '2_medium' ELSE '3_volatile' END AS grp_name,
       COUNT(*)                              AS n,
       SQRT(AVG(POW(prediction - label, 2))) AS rmse_model,
       SQRT(AVG(POW(baseline   - label, 2))) AS rmse_base,
       AVG(ABS(prediction - label))          AS mae_model,
       AVG(ABS(baseline   - label))          AS mae_base
FROM q2_base
GROUP BY grp
ORDER BY grp
"""

spark.sql(q2_view_sql)                  # creates the q2_base view
q2_month = spark.sql(q2_month_sql)      # error by month
q2_group = spark.sql(q2_group_sql)      # error by calm / medium / volatile

q2_month.show()
q2_group.show()
print("rows in q2_base:", spark.table("q2_base").count())   # expect 25441

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("task4/outputs", exist_ok=True)

# Both results are tiny (12 and 3 rows), so pandas is fine for plotting
by_month = q2_month.toPandas().set_index("month")
by_group = q2_group.toPandas().set_index("grp_name")
by_month.to_csv("task4/outputs/q2_error_by_month.csv")
by_group.to_csv("task4/outputs/q2_error_by_group.csv")

# Chart 1: error by month
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(by_month.index, by_month["rmse_model"], marker="o", label="Model (linear regression)")
ax.plot(by_month.index, by_month["rmse_base"], marker="o", label="Baseline (12-week average)")
ax.set_xlabel("Month (2017)")
ax.set_ylabel("RMSE of next-week volatility")
ax.set_title("Q2: Prediction error by month, 2017 test set")
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("task4/outputs/q2_chart_by_month.png", dpi=150)
plt.show()

# Chart 2: error by volatility group
fig, ax = plt.subplots(figsize=(7, 4))
x = range(len(by_group))
w = 0.38
ax.bar([i - w / 2 for i in x], by_group["rmse_model"], w, label="Model")
ax.bar([i + w / 2 for i in x], by_group["rmse_base"], w, label="Baseline")
ax.set_xticks(list(x))
ax.set_xticklabels([f"{g}\n(n = {n:,})" for g, n in zip(by_group.index, by_group["n"])])
ax.set_ylabel("RMSE of next-week volatility")
ax.set_title("Q2: Prediction error by stock volatility group, 2017")
ax.legend()
plt.tight_layout()
plt.savefig("task4/outputs/q2_chart_by_group.png", dpi=150)
plt.show()